<a href="https://colab.research.google.com/github/andrewgodbout/MCS-3950-F26/blob/main/Notebooks/L03-Spatial_Filtering/L3_Spatial_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L3 — Spatial Filtering: Follow-Along Examples
**MCS 3950 Computer Vision · UPEI · Fall 2026**

This notebook accompanies **Lecture 3** and provides runnable code

| Section | Topic |
|---------|-------|
| 0 | Convolution theory & key properties |
| 0b | Kernel zoo — 8 named kernels visualised |
| 1 | Convolution from scratch |
| 2 | Box (mean) filter |
| 3 | Gaussian filter |
| 4 | Bilateral filter |
| 5 | Sharpening — unsharp masking |
| 6 | Border handling |
| ✏️ | **Try at home** (Match-the-kernel + Denoising) |

> **Note:** This notebook is *not* submitted or graded.


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data as skdata

plt.rcParams.update({"figure.dpi":110, "axes.spines.top":False,
                     "axes.spines.right":False, "font.size":11})

# We use the astronaut image throughout (RGB, 512×512)
img_rgb  = skdata.astronaut()
img_gray = cv2.cvtColor(img_rgb[:,:,::-1], cv2.COLOR_BGR2GRAY)

print(f"RGB  shape: {img_rgb.shape}   dtype: {img_rgb.dtype}")
print(f"Gray shape: {img_gray.shape}  dtype: {img_gray.dtype}")

# ── Display Images ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(4.5, 2.25))

# RGB Image
ax1.imshow(img_rgb)
ax1.set_title("RGB Image")
ax1.axis("off")

# Grayscale Image
ax2.imshow(img_gray, cmap="gray")
ax2.set_title("Grayscale Image")
ax2.axis("off")

plt.tight_layout()
plt.show()


---
## 0 · Convolution Theory & Key Properties


**Slide connection** — before diving into specific filters, four properties
of convolution are worth understanding. They explain *why* the operations work
the way they do, and they come up again in edge detection (L4), scale-space
(L6), and wherever you're combining or optimising filters.

| Property | What it means |
|----------|---------------|
| **Linearity** | `filter(α·f + β·g) = α·filter(f) + β·filter(g)` |
| **Shift-invariance** | Same kernel at every pixel position — no special treatment for the centre |
| **Separability** | Some 2-D kernels split into two 1-D passes (O(k²) → O(2k) per pixel) |
| **Associativity** | Chaining two filters = convolving their kernels first, then applying once |

We'll demonstrate each one with actual NumPy/OpenCV code.


In [ ]:
# ── Property 1: Linearity ─────────────────────────────────────────────────────
# If we scale an image by α before filtering, the output scales the same way.
# If we add two images and filter, it equals filtering each and adding.

box = np.ones((9, 9), np.float32) / 81   # 9×9 box kernel

f      = img_gray.astype(np.float32)
alpha  = 1.5
g      = np.roll(img_gray, 50, axis=1).astype(np.float32)   # shifted copy as a second "image"

# Test 1: scalar multiplication
#multiply image by alpha first
lhs1 = cv2.filter2D(alpha * f, -1, box)
#apply box filter them multiply result by alpha
rhs1 = alpha * cv2.filter2D(f, -1, box)
print(f"Linearity (scalar): max |lhs − rhs| = {np.abs(lhs1 - rhs1).max():.4f}  (should be ≈ 0)")

# Test 2: additivity
#add two images togther first then apply filter
lhs2 = cv2.filter2D(f + g, -1, box)
#apply filter to each image then add the results
rhs2 = cv2.filter2D(f, -1, box) + cv2.filter2D(g, -1, box)
print(f"Linearity (additive): max |lhs − rhs| = {np.abs(lhs2 - rhs2).max():.4f}  (should be ≈ 0)")

print()
print("Why does this matter?")
print("  • RGB images: you can filter each channel independently and combine.")
print("  • Unsharp masking: mask = f - blur(f) → sharp = f + α·mask = (1+α)·f - α·blur(f)")
print("    This is just a linear combination of f and blur(f).")
print("  • Frequency analysis: the Fourier transform of (f ∗ h) = F·H (pointwise product).")
print("    Linearity is what makes this decomposition valid.")


In [ ]:
# ── Property 2: Shift-Invariance ─────────────────────────────────────────────
# Filtering at position (x,y) gives the same result regardless of where in the
# image (x,y) is.  Equivalently: filter(shift(f)) = shift(filter(f)).

box5 = np.ones((5, 5), np.float32) / 25

# Shift the image, then filter
shifted_first  = cv2.filter2D(np.roll(img_gray, 80, axis=1), -1, box5)
# Filter, then shift
filtered_first = np.roll(cv2.filter2D(img_gray, -1, box5), 80, axis=1)

# Compare a crop away from the wrap-around boundary
crop = np.s_[50:400, 100:400]
diff = np.abs(shifted_first[crop].astype(float) - filtered_first[crop].astype(float))
print(f"Shift-invariance: max difference in interior crop = {diff.max():.4f}  (should be ≈ 0)")
print()
print("Why does this matter?")
print("  • A cat in the corner of an image gets the same filter response as a")
print("    cat in the centre.  No privileged location.")
print("  • This property is exploited by CNNs: convolutional layers are weight-shared")
print("    across positions precisely because natural features are shift-invariant.")
print("  • Linearity + shift-invariance = LSI system.  For LSI systems, the")
print("    kernel fully characterises what the filter does.")


In [ ]:
# ── Property 3: Separability ─────────────────────────────────────────────────
# A 2-D kernel h(x,y) is separable if h = h_row · h_col^T (outer product).
# Gaussian is the classic example.  Box filter is also separable (trivially).
#
# Speedup: apply h_col as a 1×k vertical pass, then h_row as a k×1 horizontal
# pass.  Each pass does k multiplies/pixel → 2k total vs. k² for the full 2-D.

import time

sigma = 7
ksize = int(6*sigma+1) | 1   # must be odd

# 2-D Gaussian kernel (outer product of two 1-D kernels)
k1d   = cv2.getGaussianKernel(ksize, sigma)
k2d   = k1d @ k1d.T

print(f"Kernel size: {ksize}×{ksize}  ({ksize**2} weights)")
print(f"Separable:   2 × {ksize} = {2*ksize} weights  →  {ksize**2/(2*ksize):.1f}× fewer multiply-adds")

# Time the 2-D approach (simulated with filter2D)
N = 50
t0 = time.perf_counter()
for _ in range(N):
    r2d = cv2.filter2D(img_gray, -1, k2d)
t_2d = (time.perf_counter()-t0)/N * 1000

# Time the separable approach (OpenCV does this internally with GaussianBlur)
t0 = time.perf_counter()
for _ in range(N):
    r1d = cv2.GaussianBlur(img_gray, (ksize, ksize), sigma)
t_1d = (time.perf_counter()-t0)/N * 1000

diff = np.abs(r2d.astype(float) - r1d.astype(float)).max()
print(f"\n2-D filter2D:      {t_2d:.2f} ms/frame")
print(f"Separable GaussianBlur: {t_1d:.2f} ms/frame")
print(f"Speedup: {t_2d/t_1d:.1f}×   (max pixel diff = {diff:.2f})")

# Verify separability mathematically
k2d_reconstructed = k1d @ k1d.T
print(f"\nOuter-product reconstruction matches original: {np.allclose(k2d, k2d_reconstructed)}")
print("The Gaussian is the most important separable kernel — it's the only")
print("rotationally symmetric kernel that is also separable.")


In [ ]:
# ── Property 4: Associativity (Composability) ─────────────────────────────────
# Applying filter h₁ then h₂ is the same as applying h₁ ∗ h₂ in one pass.
# (f ∗ h₁) ∗ h₂ = f ∗ (h₁ ∗ h₂)
#
# Practical use:
#   • Pre-compute the combined kernel when the same pair of filters is applied
#     to many images (e.g., a pre-processing pipeline).
#   • Understand why blurring then sharpening can be expressed as a single kernel.

# LoG (Laplacian of Gaussian) = Gaussian blur THEN Laplacian
# Commonly used for edge detection (L4). Let's verify associativity holds.

gauss_k = cv2.getGaussianKernel(11, 2) @ cv2.getGaussianKernel(11, 2).T
lap_k   = np.array([[0, 1, 0],[1,-4, 1],[0, 1, 0]], dtype=np.float32)

# Two-pass approach
two_pass = cv2.filter2D(cv2.filter2D(img_gray, -1, gauss_k), -1, lap_k)

# One-pass: convolve the kernels first, then apply
combined_k = cv2.filter2D(gauss_k, -1, lap_k)   # convolve the two kernels
one_pass   = cv2.filter2D(img_gray, -1, combined_k)

max_err = np.abs(two_pass.astype(float) - one_pass.astype(float)).max()
print(f"Associativity: max |two-pass − one-pass| = {max_err:.4f}  (should be ≈ 0)")

# Visualise the combined kernel (Laplacian of Gaussian)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, k, title in zip(axes,
    [gauss_k, lap_k, combined_k],
    ["Gaussian kernel (11×11)", "Laplacian kernel (3×3)", "LoG = Gaussian ∗ Laplacian"]):
    ax.imshow(k, cmap='RdBu_r')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Associativity demo: apply G then L  =  apply (G ∗ L) directly", fontsize=12)
plt.tight_layout()
plt.show()

print("\nLoG kernel visualisation:")
print("  Centre: negative (detects rapid change).")
print("  Surround: positive (suppresses uniform regions).")
print("  We will use this directly in L4 — edge detection.")


---
## 0b · Kernel Zoo — 8 Named Kernels


**Slide connection** — the formula `(f ∗ h)[x,y] = Σ f · h` works for any
kernel `h`. Different kernels do wildly different things to an image.

Below we define 8 classic kernels, print them, and apply them all to the same
image so you can build an intuition for what to *expect* from a kernel before
you calculate anything.

> **Reading a kernel visually:**
> - **Positive weights** → those neighbours are included (added) in the output
> - **Negative weights** → those neighbours are subtracted (edge detection)
> - **All weights equal** → simple average (blur)
> - **Centre positive, surround negative** → detect local peaks (sharpen)
> - **Asymmetric** → directional response (Sobel, emboss, motion)


In [ ]:
# ── Define the kernel zoo ─────────────────────────────────────────────────────
kernels = {}

kernels["Identity"] = np.array([
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
], dtype=np.float32)

kernels["Box 3×3"] = np.ones((3,3), dtype=np.float32) / 9.0

kernels["Gaussian\n(approx)"] = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1],
], dtype=np.float32) / 16.0

kernels["Sharpen"] = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0],
], dtype=np.float32)

kernels["Laplacian\n(edge detect)"] = np.array([
    [0,  1, 0],
    [1, -4, 1],
    [0,  1, 0],
], dtype=np.float32)

kernels["Sobel X\n(horiz edges)"] = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

kernels["Emboss"] = np.array([
    [-2, -1, 0],
    [-1,  1, 1],
    [ 0,  1, 2],
], dtype=np.float32)

# 9×9 horizontal motion blur
motion = np.zeros((9, 9), dtype=np.float32)
motion[4, :] = 1.0 / 9.0
kernels["Motion blur\n(horiz)"] = motion

# ── Print each kernel ─────────────────────────────────────────────────────────
for name, k in kernels.items():
    label = name.replace("\n", " ")
    print(f"── {label} ──")
    # Format with alignment

    for row in k:
        fmt = "  ".join(f"{v:+6.3f}" for v in row) if k.dtype == np.float32 else ""
        print("  " + "  ".join(f"{v:+6.3f}" for v in row))
    print()


In [ ]:
# ── Apply all kernels to the same image ──────────────────────────────────────
names  = list(kernels.keys())
kvals  = list(kernels.values())
n      = len(names)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, name, k in zip(axes.ravel(), names, kvals):
    result = cv2.filter2D(img_gray, cv2.CV_32F, k)
    # For edge kernels (Laplacian, Sobel) show absolute value, normalised
    display = result
    if result.min() < 0:
        display = np.abs(result)
    display = np.clip(display, 0, None)
    vmax = display.max() if display.max() > 0 else 1
    ax.imshow(display / vmax, cmap='gray', vmin=0, vmax=1)
    ax.set_title(name, fontsize=11)
    ax.axis('off')

plt.suptitle("The same image filtered with 8 different kernels", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Kernel weight visualisation ───────────────────────────────────────────────
# Colour map: blue=negative weights, red=positive weights, white=zero.
# This shows the 'shape' of what each kernel is responding to.

small_kernels = {k:v for k,v in kernels.items() if v.shape[0] <= 3}

fig, axes = plt.subplots(1, len(small_kernels), figsize=(14, 2.5))
for ax, (name, k) in zip(axes, small_kernels.items()):
    vmax = np.abs(k).max()
    im = ax.imshow(k, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(name.replace("\n", "\n"), fontsize=10)
    # Annotate each cell with its value
    for r in range(k.shape[0]):
        for c in range(k.shape[1]):
            ax.text(c, r, f"{k[r,c]:.2f}", ha='center', va='center',
                    fontsize=9, color='black')
    ax.axis('off')

plt.suptitle("Kernel weights visualised  (blue = negative, red = positive)", fontsize=12)
plt.tight_layout()
plt.show()

print("Observation:")
print("  Identity  — single 1 at centre → pass-through, nothing changes.")
print("  Box/Gauss — all positive, sum to 1 → weighted average → blur.")
print("  Sharpen   — centre = 5, neighbours = −1; subtracts the local average → edges enhanced.")
print("  Laplacian — centre = −4, neighbours = +1; detects where values change rapidly → edges.")
print("  Sobel X   — left column = negative, right = positive → detects left-to-right brightness change.")
print("  Emboss    — asymmetric; produces a directional '3D' relief effect.")


---
## 1 · Convolution from Scratch


**Slide connection:** the formula
`(f ∗ h)[x,y] = Σᵢ Σⱼ f[x+i, y+j] · h[i,j]`
applied at every pixel position.

We implement a slow (but transparent) version to make the operation concrete,
then verify it gives the same result as `cv2.filter2D`.


In [ ]:
def convolve2d_slow(img, kernel):
    """
    Cross-correlation (what OpenCV calls 'convolution').
    img    : 2-D uint8 array
    kernel : 2-D float array — should sum to 1 for a blur kernel
    Returns float64 array, same shape as img.
    """
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    # Reflect-pad so output is the same size as input
    padded = np.pad(img.astype(np.float64), ((pad_h, pad_h), (pad_w, pad_w)),
                    mode='reflect')
    out = np.zeros_like(img, dtype=np.float64)
    for r in range(img.shape[0]):
        for c in range(img.shape[1]):
            patch = padded[r:r+kh, c:c+kw]
            out[r, c] = (patch * kernel).sum()
    return out


In [ ]:
# 3×3 box kernel — each weight = 1/9
box_kernel = np.ones((3, 3), dtype=np.float32) / 9.0
print("Box kernel:")
print(np.round(box_kernel, 3))

# Apply our slow version to a small crop for speed
crop = img_gray[:64, :64]
result_slow = convolve2d_slow(crop, box_kernel)

# Apply OpenCV's version (fast, same math)
result_cv2  = cv2.filter2D(crop, -1, box_kernel)

# Compare
max_diff = np.abs(result_slow - result_cv2.astype(np.float64)).max()
print(f"\nMax pixel difference (slow vs cv2.filter2D): {max_diff:.4f}")
print("They match!" if max_diff < 1.0 else "Mismatch — check the kernel.")


In [ ]:
# Worked example from slides — a specific 5×5 patch
patch = np.array([
    [ 80,  95, 110, 120,  90],
    [100,  90, 120, 130, 110],
    [ 85, 100, 110, 140, 120],
    [ 95, 110, 130, 110,  95],
    [ 70,  85, 100,  90,  80],
], dtype=np.float32)

inner_3x3 = patch[1:4, 1:4]   # the highlighted 3×3 region
output_value = inner_3x3.mean()

print("Inner 3×3 patch (highlighted in slide 3):")
print(inner_3x3.astype(int))
print(f"\nBox filter output at centre pixel: {inner_3x3.sum():.0f} / 9 = {output_value:.1f}")
print(f"Original centre value: {patch[2, 2]:.0f}")
print(f"Change: {patch[2, 2]:.0f} → {output_value:.1f}  (small — neighbourhood is fairly uniform)")


---
## 2 · Box Filter


**Slide connection:** uniform kernel — each neighbour contributes equally.
Larger kernel = more blur, but a visible "box" artifact near hard edges.


In [ ]:
# Apply box filter at several kernel sizes
ks_list = [3, 9, 21]
results  = [cv2.blur(img_gray, (k, k)) for k in ks_list]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original")
for ax, k, r in zip(axes[1:], ks_list, results):
    ax.imshow(r, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"Box k={k}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Zoom into an edge to see the 'box' artifact vs Gaussian (preview)
row_idx = 200    # horizontal slice through the image at this row
x = np.arange(img_gray.shape[1])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x, img_gray[row_idx], label='Original',      lw=1.2, color='#1A3A5C')
ax.plot(x, cv2.blur(img_gray, (21,21))[row_idx],
        label='Box k=21',  lw=1.5, color='#C8962C', linestyle='--')
ax.plot(x, cv2.GaussianBlur(img_gray, (21,21), 0)[row_idx],
        label='Gaussian k=21', lw=1.5, color='#1D7A8C', linestyle='-.')
ax.set_title(f"Intensity profile at row {row_idx}")
ax.set_xlabel("Column (pixel)")
ax.set_ylabel("Intensity")
ax.legend()
plt.tight_layout()
plt.show()

print("Notice: Gaussian profile is smoother — no flat-top 'box' shape in the transition zone.")


---
## 3 · Gaussian Filter


**Slide connection:** centre-weighted average controlled by σ.
`cv2.GaussianBlur(img, (0,0), sigmaX)` — passing `(0,0)` lets OpenCV
compute the kernel size from σ automatically.

Key property: **separable** — applied as two 1-D passes (rows then columns),
so it's O(k) per pixel not O(k²).


In [ ]:
sigmas = [1, 2, 5, 10]
blurred = [cv2.GaussianBlur(img_gray, (0, 0), s) for s in sigmas]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original")
for ax, s, b in zip(axes[1:], sigmas, blurred):
    ax.imshow(b, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"σ = {s}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Visualise the 2-D Gaussian kernel itself
sigma = 3
ksize = int(6 * sigma + 1) | 1   # round up to odd
kern1d = cv2.getGaussianKernel(ksize, sigma)
kern2d = kern1d @ kern1d.T        # outer product → 2-D kernel

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(kern2d, cmap='hot')
axes[0].set_title(f"2-D Gaussian kernel  σ={sigma}  ({ksize}×{ksize})")
plt.colorbar(im, ax=axes[0])

axes[1].plot(kern1d.ravel(), color='#1A3A5C', lw=2)
axes[1].set_title("1-D cross-section (separable!)")
axes[1].set_xlabel("Kernel position")
axes[1].set_ylabel("Weight")
plt.tight_layout()
plt.show()

print(f"Kernel sums to: {kern2d.sum():.6f}  (should be ≈ 1.0)")
print(f"Centre weight:  {kern2d[ksize//2, ksize//2]:.4f}")
print(f"Corner weight:  {kern2d[0, 0]:.6f}  (much smaller → smooth roll-off)")


---
## 4 · Bilateral Filter


**Slide connection:**
`W(p,q) = Gₛ(‖p−q‖) × Gᵣ(|f(p)−f(q)|)`

The range Gaussian `Gᵣ` gives near-zero weight to pixels on the other side
of an edge — so the edge acts as a barrier the filter cannot cross.


In [ ]:
# Add synthetic Gaussian noise so we can compare denoising methods
rng   = np.random.default_rng(42)
noise = rng.normal(0, 25, img_gray.shape).astype(np.float32)
noisy = np.clip(img_gray.astype(np.float32) + noise, 0, 255).astype(np.uint8)

gauss    = cv2.GaussianBlur(noisy, (0, 0), sigmaX=3)
bilateral = cv2.bilateralFilter(noisy, d=9, sigmaColor=75, sigmaSpace=75)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(axes,
    [img_gray, noisy, gauss, bilateral],
    ["Clean original", "Noisy (σ=25)", "Gaussian σ=3", "Bilateral d=9 σ=75"]):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Zoom into an edge region to see the key difference
r0, r1, c0, c1 = 120, 250, 280, 380   # crop around suit/background boundary

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(axes,
    [noisy[r0:r1, c0:c1], gauss[r0:r1, c0:c1], bilateral[r0:r1, c0:c1]],
    ["Noisy", "Gaussian (edges blurred)", "Bilateral (edges preserved)"],
    ):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.suptitle("Edge region zoom — bilateral preserves the boundary", y=1.02)
plt.tight_layout()
plt.show()

# Measure noise reduction
flat = np.s_[350:420, 50:120]
for name, im in [("Noisy", noisy), ("Gaussian", gauss), ("Bilateral", bilateral)]:
    rms = np.sqrt(np.mean((img_gray[flat].astype(float) - im[flat].astype(float))**2))
    print(f"{name:12s}  RMS error in flat region: {rms:.2f}")


In [ ]:
# Bilateral on colour — the 'beauty filter' effect
img_bgr      = img_rgb[:, :, ::-1]   # RGB -> BGR for OpenCV
bilateral_col = cv2.bilateralFilter(img_bgr, d=15, sigmaColor=80, sigmaSpace=80)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_rgb)
axes[0].set_title("Original")
axes[1].imshow(bilateral_col[:, :, ::-1])
axes[1].set_title("Bilateral d=15 σ=80\n(portrait-mode smoothing effect)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Apply bilateral filter multiple times to increase abstraction
cartoon_img = img_rgb.copy()
for _ in range(10):
    cartoon_img = cv2.bilateralFilter(cartoon_img, d=9, sigmaColor=50, sigmaSpace=50)

# Optional: Quantize colors or overlay detected edges to finish cartoon effect
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
edges = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                              cv2.THRESH_BINARY, blockSize=9, C=9)
cartoon_final = cv2.bitwise_and(cartoon_img, cartoon_img, mask=edges)
plt.tight_layout()
plt.axis('off')
plt.imshow(cartoon_img)

---
## 5 · Sharpening — Unsharp Masking


**Slide connection:**

```
mask  = f  −  Gσ * f          # high-frequency edge signal
sharp = f  +  α × mask        # add it back, scaled by strength α
```

Equivalently: `sharp = (1+α)·f − α·(Gσ*f)`


In [ ]:
def unsharp_mask(img, sigma=2.0, alpha=1.0):
    """
    Sharpen an image using unsharp masking.
    img   : uint8 array (grayscale or colour)
    sigma : controls how wide the 'edge' signal is detected
    alpha : sharpening strength (0.5 = subtle, 2.0 = strong)
    """
    blurred = cv2.GaussianBlur(img, (0, 0), sigma)
    mask    = img.astype(np.float32) - blurred.astype(np.float32)
    sharp   = np.clip(img.astype(np.float32) + alpha * mask, 0, 255)
    return sharp.astype(np.uint8)


# Compare different sharpening strengths on grayscale
alphas = [0.5, 1.0, 2.0, 4.0]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original")
for ax, a in zip(axes[1:], alphas):
    ax.imshow(unsharp_mask(img_gray, sigma=2, alpha=a), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"α = {a}")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Show the mask itself — what gets added back
blurred_for_mask = cv2.GaussianBlur(img_gray, (0, 0), 2)
mask = img_gray.astype(np.float32) - blurred_for_mask.astype(np.float32)
mask_display = np.clip(mask + 128, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("f  (original)")
axes[1].imshow(blurred_for_mask, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("Gσ*f  (blurred, σ=2)")
axes[2].imshow(mask_display, cmap='gray', vmin=0, vmax=255)
axes[2].set_title("mask = f − Gσ*f\n(128 = zero; bright/dark = edges)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()
print("The mask is zero in flat areas and non-zero at edges — exactly the detail we add back.")


In [ ]:
# WARNING: sharpening amplifies noise
noisy_sharp = unsharp_mask(noisy, sigma=2, alpha=1.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(axes,
    [img_gray, noisy, noisy_sharp],
    ["Clean original", "Noisy (σ=25)", "Noisy + sharpen α=1.5\n← amplified noise!"]):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
print("Workflow: denoise first (bilateral or Gaussian), THEN sharpen.")


---
## 6 · Border Handling


**Slide 8 connection:** when the kernel extends past the image edge, OpenCV
needs to know what values to use. The `borderType` parameter controls this.


In [ ]:
corner = img_gray[:60, :80]
k = np.ones((11, 11), dtype=np.float32) / 121   # large kernel so the effect is obvious

border_modes = [
    (cv2.BORDER_CONSTANT,    "BORDER_CONSTANT (0)"),
    (cv2.BORDER_REFLECT,     "BORDER_REFLECT"),
    (cv2.BORDER_REFLECT_101, "BORDER_REFLECT_101 (default)"),
    (cv2.BORDER_REPLICATE,   "BORDER_REPLICATE"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (mode, name) in zip(axes, border_modes):
    result = cv2.filter2D(corner, -1, k, borderType=mode)
    ax.imshow(result, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name, fontsize=9.5)
    ax.axis('off')
plt.suptitle("Same 11×11 box filter on a corner crop — only border handling differs", y=1.02)
plt.tight_layout()
plt.show()

print("BORDER_CONSTANT: notice the dark fringe at the top-left corner.")
print("BORDER_REFLECT_101: smooth, no visible fringe — that is why it is the default.")


---
## ✏️ Try at Home — Part A: Match the Kernel


**Estimated time: 5 minutes**

Below you'll see **five filtered versions** of the same image, labelled
Output 1–5. Below that are **five kernels** labelled A–E (in scrambled order).

Your task: figure out which kernel produced which output **without running
any code**. Reason from the kernel weights alone.

Once you have a guess, run the **verification cell** to check.

### The kernels (scrambled)

| Label | Kernel |
|-------|--------|
| **A** | `[[-1,0,1],[-2,0,2],[-1,0,1]]` |
| **B** | `ones(15,15) / 225` |
| **C** | `[[0,1,0],[1,-4,1],[0,1,0]]` |
| **D** | `[[-2,-1,0],[-1,1,1],[0,1,2]]` |
| **E** | `[[0,-1,0],[-1,5,-1],[0,-1,0]]` |

*Hint: ask yourself what each kernel is summing. Does the output get brighter,
darker, or show edges? Is there a preferred direction?*


In [ ]:
# ── Generate the five mystery outputs ────────────────────────────────────────
mystery_kernels = {
    "A": np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32),
    "B": np.ones((15,15), dtype=np.float32) / 225.0,
    "C": np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32),
    "D": np.array([[-2,-1,0],[-1,1,1],[0,1,2]], dtype=np.float32),
    "E": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
}

# Scrambled assignment: kernel → output number (HIDDEN from student)
_assignment = {"A":3, "B":1, "C":5, "D":4, "E":2}
# Output N uses kernel _assignment inverse
_inv = {v:k for k,v in _assignment.items()}

outputs = {}
for out_num in range(1, 6):
    k = mystery_kernels[_inv[out_num]]
    raw = cv2.filter2D(img_gray, cv2.CV_32F, k)
    # Absolute value + normalise for display (edge kernels go negative)
    disp = np.abs(raw)
    outputs[out_num] = np.clip(disp / disp.max() * 255, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
for ax, (num, im) in zip(axes, outputs.items()):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"Output {num}", fontsize=13, fontweight='bold')
    ax.axis('off')
plt.suptitle("Match each output to a kernel (A–E). Reasoning first, then check!", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ── Your answer — fill in your guesses here ───────────────────────────────────
# Edit this dictionary: output_number → your kernel label ('A'–'E')
my_answers = {
    1: '?',   # replace '?' with your guess
    2: '?',
    3: '?',
    4: '?',
    5: '?',
}

# ── Verification ─────────────────────────────────────────────────────────────
correct = {"A":3, "B":1, "C":5, "D":4, "E":2}   # output_num → correct kernel label flipped
answers_flipped = {v:k for k,v in correct.items()}   # output → correct label

if '?' in my_answers.values():
    print("Fill in all five guesses first (replace '?' with A, B, C, D, or E).")
else:
    score = 0
    for out_num in range(1, 6):
        guess   = my_answers[out_num].upper()
        correct_label = answers_flipped[out_num]
        ok = guess == correct_label
        score += ok
        status = "✓" if ok else "✗"
        print(f"Output {out_num}: you said {guess!r}  →  correct: {correct_label!r}  {status}")
    print(f"\nScore: {score}/5")
    if score == 5:
        print("Perfect! You have good kernel intuition.")
    elif score >= 3:
        print("Good — revisit the ones you missed: look at the kernel weights again.")
    else:
        print("Try re-reading the 'Reading a kernel visually' tips in Section 0b,")
        print("then come back and try again without looking at the answers.")


---
## ✏️ Try at Home — Part B: Noisy Image Denoising Comparison


**Estimated time: 5–10 minutes**

Complete the two functions below, then run the test cell.

### Task

1. **`add_gaussian_noise(img, sigma)`** — add zero-mean Gaussian noise to a
   `uint8` image. Don't forget to clip and convert back to `uint8`.

2. **`compare_denoising(img, noise_sigma, gauss_sigma, bilat_d, bilat_sigma)`**
   — add noise, then apply both a Gaussian filter and a bilateral filter.
   Return `(noisy, gauss_denoised, bilat_denoised)`.

### What to look for

Run the test cell and compare the three outputs side by side:

- In **flat, smooth** regions: both methods should reduce noise similarly.
- At **sharp edges**: Gaussian will soften the edge; bilateral should keep it sharp.

### Bonus (optional)

Compute the RMS error between each denoised output and the clean original
for a flat patch and an edge patch. Which method wins on each?


In [ ]:
def add_gaussian_noise(img, sigma=30):
    """
    Add zero-mean Gaussian noise to a uint8 image.

    Parameters
    ----------
    img   : np.ndarray, dtype uint8
    sigma : standard deviation of the noise (larger = more noise)

    Returns
    -------
    np.ndarray, dtype uint8, same shape as img
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────


    pass   # replace this line


def compare_denoising(img, noise_sigma=30, gauss_sigma=2,
                      bilat_d=9, bilat_sigma=75):
    """
    Add noise then denoise with two methods.

    Returns
    -------
    (noisy, gauss_denoised, bilat_denoised) — all uint8 arrays
    """
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────

    pass   # replace this line


In [ ]:
# ── Test cell — run after completing the functions above ─────────────────────
result = compare_denoising(img_gray, noise_sigma=35, gauss_sigma=3,
                           bilat_d=9, bilat_sigma=75)

if result is None:
    print("compare_denoising() returned None — implement the function first.")
else:
    noisy_out, gauss_out, bilat_out = result

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, im, title in zip(axes,
        [img_gray, noisy_out, gauss_out, bilat_out],
        ["Clean original", "Noisy (σ=35)", "Gaussian σ=3", "Bilateral d=9 σ=75"]):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.suptitle("Denoising comparison", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # Quick edge sharpness test
    edge_row = np.s_[180, 270:340]
    print("\nIntensity profile across an edge (column 270–340, row 180):")
    for name, im in [("Original", img_gray), ("Gaussian", gauss_out), ("Bilateral", bilat_out)]:
        profile = im[edge_row]
        gradient = int(np.abs(np.diff(profile.astype(int))).max())
        print(f"  {name:12s}  max gradient = {gradient:3d}  (higher = sharper edge)")
